In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from config import get_engine
import joblib


In [2]:
OUTPUT_DIR = Path('output')
engine = get_engine()

In [3]:
import pandas as pd

churn_df = pd.read_csv("customer_churn_probability.csv")
revenue_df = pd.read_csv("customer_revenue_prediction.csv")
segment_df = pd.read_csv("customer_segments_labeled.csv")

customer_profile = churn_df.merge(
    revenue_df,
    on="Customer_ID"
).merge(
    segment_df,
    on="Customer_ID"
)

customer_profile.to_csv(
    "customer_intelligence_profile.csv",
    index=False
)

print(customer_profile.head())

   Customer_ID  Churn_Probability Dataset_x  Predicted_Revenue Dataset_y  \
0        14738           0.000043     Train           1926.475     Train   
1        14738           0.000043     Train           1926.475     Train   
2        14738           0.000043     Train           1926.475      Test   
3        14738           0.000043     Train           1926.475      Test   
4        14554           0.000237     Train          13712.090     Train   

   Cluster                       Cluster_Name  
0        2        High-Value Growth Customers  
1        2        High-Value Growth Customers  
2        2        High-Value Growth Customers  
3        2        High-Value Growth Customers  
4        1  High-Revenue Enterprise Customers  


In [4]:
customer_profile["Risk_Level"] = customer_profile["Churn_Probability"].apply(
    lambda x: "High Risk" if x > 0.7 else "Low Risk"
)

customer_profile["Revenue_Category"] = customer_profile["Predicted_Revenue"].apply(
    lambda x: "High Revenue" if x > 5000 else "Low Revenue"
)

In [5]:
customer_profile.head()

,Customer_ID,Churn_Probability,Dataset_x,Predicted_Revenue,Dataset_y,Cluster,Cluster_Name,Risk_Level,Revenue_Category
0,14738,0.000043,Train,1926.475,Train,2,High-Value Growth Customers,Low Risk,Low Revenue
1,14738,0.000043,Train,1926.475,Train,2,High-Value Growth Customers,Low Risk,Low Revenue
2,14738,0.000043,Train,1926.475,Test,2,High-Value Growth Customers,Low Risk,Low Revenue
3,14738,0.000043,Train,1926.475,Test,2,High-Value Growth Customers,Low Risk,Low Revenue
4,14554,0.000237,Train,13712.090,Train,1,High-Revenue Enterprise Customers,Low Risk,High Revenue


In [6]:
customer_profile.shape

(5798, 9)

In [7]:
customer_profile.to_json("customer_intelligence_profile.json")

In [8]:
customer_profile.duplicated().sum()

np.int64(666)

In [9]:
# Remove duplicate rows if needed
df_unique = customer_profile.drop_duplicates()

# Select required columns
json_data = df_unique[[
    'Customer_ID',
    'Churn_Probability',
    'Predicted_Revenue',
    'Cluster',
    'Cluster_Name',
    'Risk_Level',
    'Revenue_Category'
]].to_dict(orient='records')

# Save JSON file
import json

with open('customer_analytics.json', 'w') as f:
    json.dump(json_data, f, indent=4)

print(json_data[:5])

[{'Customer_ID': 14738, 'Churn_Probability': 4.3449778e-05, 'Predicted_Revenue': 1926.475, 'Cluster': 2, 'Cluster_Name': 'High-Value Growth Customers', 'Risk_Level': 'Low Risk', 'Revenue_Category': 'Low Revenue'}, {'Customer_ID': 14738, 'Churn_Probability': 4.3449778e-05, 'Predicted_Revenue': 1926.475, 'Cluster': 2, 'Cluster_Name': 'High-Value Growth Customers', 'Risk_Level': 'Low Risk', 'Revenue_Category': 'Low Revenue'}, {'Customer_ID': 14554, 'Churn_Probability': 0.00023698453, 'Predicted_Revenue': 13712.09, 'Cluster': 1, 'Cluster_Name': 'High-Revenue Enterprise Customers', 'Risk_Level': 'Low Risk', 'Revenue_Category': 'High Revenue'}, {'Customer_ID': 13237, 'Churn_Probability': 0.00045998197, 'Predicted_Revenue': 222.54, 'Cluster': 0, 'Cluster_Name': 'Mid-Tier Active Customers', 'Risk_Level': 'Low Risk', 'Revenue_Category': 'Low Revenue'}, {'Customer_ID': 11773, 'Churn_Probability': 6.4438704e-05, 'Predicted_Revenue': 7322.035, 'Cluster': 1, 'Cluster_Name': 'High-Revenue Enterprise

In [15]:
import pandas as pd
import json
from google import genai

# Load Customer Intelligence Profile
customer_profile = pd.read_csv("customer_intelligence_profile.csv")

# Create Risk Level if missing
if "Risk_Level" not in customer_profile.columns:
    customer_profile["Risk_Level"] = customer_profile["Churn_Probability"].apply(
        lambda x: "High Risk" if x > 0.7 else "Low Risk"
    )

# Create Revenue Category if missing
if "Revenue_Category" not in customer_profile.columns:
    customer_profile["Revenue_Category"] = customer_profile["Predicted_Revenue"].apply(
        lambda x: "High Revenue" if x > 5000 else "Low Revenue"
    )

# Business Summary
summary = {
    "Total_Customers": int(len(customer_profile)),
    "Average_Churn_Probability": round(
        customer_profile["Churn_Probability"].mean(), 4
    ),
    "High_Risk_Customers": int(
        len(customer_profile[
            customer_profile["Risk_Level"] == "High Risk"
        ])
    ),
    "Low_Risk_Customers": int(
        len(customer_profile[
            customer_profile["Risk_Level"] == "Low Risk"
        ])
    ),
    "Total_Predicted_Revenue": round(
        customer_profile["Predicted_Revenue"].sum(), 2
    ),
    "Average_Predicted_Revenue": round(
        customer_profile["Predicted_Revenue"].mean(), 2
    ),
    "Cluster_Distribution":
        customer_profile["Cluster_Name"].value_counts().to_dict(),
    "Revenue_Distribution":
        customer_profile["Revenue_Category"].value_counts().to_dict(),
    "Risk_Distribution":
        customer_profile["Risk_Level"].value_counts().to_dict()
}

print(summary)

{'Total_Customers': 5798, 'Average_Churn_Probability': np.float64(0.1149), 'High_Risk_Customers': 663, 'Low_Risk_Customers': 5135, 'Total_Predicted_Revenue': np.float64(30044855.1), 'Average_Predicted_Revenue': np.float64(5181.93), 'Cluster_Distribution': {'High-Value Growth Customers': 1787, 'Mid-Tier Active Customers': 1448, 'At-Risk Low Engagement Customers': 1440, 'High-Revenue Enterprise Customers': 1123}, 'Revenue_Distribution': {'Low Revenue': 3914, 'High Revenue': 1884}, 'Risk_Distribution': {'Low Risk': 5135, 'High Risk': 663}}


In [23]:
import json

with open("business_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

In [ ]:
from google import genai
import os

client = genai.Client(
    api_key="Gen_Api_Key_Here"  # safer than hardcoding
)

In [20]:
prompt = f"""
You are a Chief Business Intelligence Officer.

Analyze the following Customer Intelligence Summary:

{json.dumps(summary, indent=2)}

Provide:

1. Executive Summary
2. Customer Churn Analysis
3. Revenue Forecast Analysis
4. Cluster Performance Analysis
5. Retention Strategy Recommendations
6. Revenue Growth Opportunities
7. Business Risks
8. Action Plan for Management

Provide professional business insights.
"""

In [21]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

## Customer Intelligence Analysis: Q4 2023 Executive Briefing

**To:** Executive Leadership Team
**From:** Chief Business Intelligence Officer (CBIO)
**Date:** October 26, 2023
**Subject:** Comprehensive Analysis of Current Customer Intelligence Summary

---

### 1. Executive Summary

Our current customer intelligence snapshot reveals a robust customer base of 5,798, with a predicted total revenue of over $30 million. However, underlying these figures are critical insights that demand strategic attention. While the majority of our customers are classified as low risk, a significant segment (663 customers, representing 11.4% of the base) is at high risk of churn, mirroring our average churn probability. A substantial portion of our customer base (3,914 customers) falls into the "Low Revenue" category, indicating significant untapped potential. Our customer segmentation highlights distinct opportunities and threats, particularly with the "At-Risk Low Engagement Customers" cluster needing